# Notebook 01 — Inter-Run Consistency Analysis

**Purpose:** Quantify inter-run agreement between two independent LLM extraction
passes (v11 vs v11_rerun1) across 64 trials. Uses the common pipeline functions
in `src/` with hand-verified metrics per data-type bucket.

**Data Flow:**

| Direction | File | From / To |
| :--------- | :---- | :--------- |
| Input | `data/raw/copd_v11.jsonl` | LLM extraction run A |
| Input | `data/raw/copd_v11_rerun1.jsonl` | LLM extraction run B |
| Input | `data/raw/field_validation_mapping.csv` | Field tier + bucket labels |
| Output | `outputs/tables/table_S1_consistency.csv` | Thesis Appendix |
| Output | `outputs/tables/pipeline/inter_run_ambiguous_flags.csv` | Feeds N03 (ambiguous PROGRESS-Plus classification) |

**Prerequisite notebooks:** None (standalone validation).

**Index:**

| Section | What it does |
| :------- | :------------ |
| 0 | Setup — paths, imports |
| 1 | Validate run consistency (structure + counts) |
| 2 | Compute per-field agreement (bucket-specific stats) |
| 3 | Build and save per-field results table (sorted by bucket) |
| 4 | Per-trial ambiguous flags (one-NA cases, feeds N03) |

> **Shared code:** Uses `src/loaders.py` (LLM runs, mapping table),
> `src/normalization.py` (per data-type bucket), and `src/agreement.py`
> (bucket-specific agreement metrics: Gwet's AC1, ICC, token F1, Jaccard).

**Notebook-specific notes:**

- **69 fields mapped, 59 analyzed:** 8 `needs_discussion_*` helper fields are
  marked `EXCLUDED` in the mapping (they flag ambiguous extractions and have no
  data counterpart in the extraction runs). 2 additional fields are EXCLUDED by
  type (free-text only). 59 actual data fields are compared.
- **Mapping file:** `field_validation_mapping.csv` classifies each field into a
  `data_type_bucket` (boolean, categorical, numerical, structured_string_array,
  free_text, EXCLUDED) and assigns a `tier1_prior_extraction` tier. To
  reclassify a field, edit the CSV and re-run this notebook.
- **Helper field convention:** All `needs_discussion_*` fields follow
  `needs_discussion_{topic}` (bool) + `needs_discussion_{topic}_explanation`
  (str), shared across all extraction schemas (COPD, CVD, DM2).


## Section 0: Setup

In [1]:
# ── Setup ────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

ROOT = Path().resolve().parent if Path().resolve().name == "notebooks"          else Path().resolve()
sys.path.insert(0, str(ROOT))

# ── External libraries ───────────────────────────────────────────────────────
import pandas as pd

# ── Project modules (src/) ───────────────────────────────────────────────────
#   src/loaders      — load LLM runs, mapping table, prior extraction, gold standard
#   src/normalization — normalize values per data-type bucket
#   src/agreement     — bucket-specific inter-run agreement metrics

from src.analysis.loaders import load_extraction_run, load_mapping_table
from src.analysis.normalization import normalize_value
from src.analysis.agreement import compute_agreement

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR = ROOT / "data" / "raw"
RUN_A_PATH = DATA_DIR / "copd_v11.jsonl"
RUN_B_PATH = DATA_DIR / "copd_v11_rerun1.jsonl"
MAPPING_TABLE_PATH = DATA_DIR / "field_validation_mapping.csv"

OUTPUT_DIR = ROOT / "outputs" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "pipeline").mkdir(parents=True, exist_ok=True)

llm_run = load_extraction_run(RUN_A_PATH)
run_b = load_extraction_run(RUN_B_PATH)
mapping = load_mapping_table(MAPPING_TABLE_PATH)

EXCLUDED_COV_NRS = {"5108"}  # >30% non-COPD participants
llm_run = llm_run[~llm_run['cov_nr'].isin(EXCLUDED_COV_NRS)]
run_b = run_b[~run_b['cov_nr'].isin(EXCLUDED_COV_NRS)]

print(f"Run A: {not llm_run.empty}")
print(f"Run B: {not run_b.empty}")
print(f"Mapping: {not mapping.empty}")
print(f"After excluding {EXCLUDED_COV_NRS}: {llm_run['cov_nr'].nunique()} trials, {len(llm_run)} arm-rows")


Run A: True
Run B: True
Mapping: True
After excluding {'5108'}: 64 trials, 8040 arm-rows


## Section 1: Validate run consistency

Check that both runs have matching structure before proceeding with
per-field analysis.


In [2]:
# Validate both runs have matching structure.
if len(llm_run) != len(run_b):
    print(f"WARNING: Run A and B have different value counts: {len(llm_run)} vs {len(run_b)}")
if llm_run["cov_nr"].nunique() != run_b["cov_nr"].nunique():
    print(f"WARNING: Run A and B have different trial counts: {llm_run["cov_nr"].nunique()} vs {run_b["cov_nr"].nunique()}")
print(f"Run A: {len(llm_run)} values, trials={llm_run['cov_nr'].nunique()}")
print(f"Run B: {len(run_b)} values, trials={run_b['cov_nr'].nunique()}")
print(f"Mapping fields: {len(mapping)}")


Run A: 8040 values, trials=64
Run B: 8040 values, trials=64
Mapping fields: 69


## Section 2: Compute per-field agreement

For each non-EXCLUDED field, pair the two runs' values, normalize them,
and compute agreement using bucket-specific statistics.

The mapping CSV has 69 fields but 8 are marked `EXCLUDED` — the
`needs_discussion_*` helper fields that flag ambiguous extractions for
manual review. They have no data counterpart in the extraction runs
to compare, so they are skipped. 61 actual data fields are analyzed.


In [3]:
# For each non-excluded field, normalize values and compute agreement.
# boolean/categorical → Gwet's AC1, numerical → ICC(2,1),
# structured_array → key Jaccard + value agreement, free_text → token F1.
# Summary cell.
results = []
per_trial_na = []

for field_name, row in mapping.iterrows():
    bucket = row['data_type_bucket']
    if bucket == 'EXCLUDED':
        continue

    a_vals = (
            llm_run[llm_run['field_name'] == field_name]
            [['cov_nr', 'arm', 'value']]
    )
    a_vals = a_vals.rename(columns={'value': 'value_a'})
    b_vals = (
            run_b[run_b['field_name'] == field_name]
            [['cov_nr', 'arm', 'value']]
    )
    b_vals = b_vals.rename(columns={'value': 'value_b'})
    merged = a_vals.merge(b_vals, on=['cov_nr', 'arm'], how='inner')

    paired = []
    for _, mr in merged.iterrows():
        norm_a = normalize_value(mr['value_a'], bucket)
        norm_b = normalize_value(mr['value_b'], bucket)
        paired.append((norm_a, norm_b, mr['cov_nr'], mr['arm']))

        is_na_a = norm_a is None
        is_na_b = norm_b is None
        if is_na_a != is_na_b:
            per_trial_na.append({
                'field_name': field_name, 'cov_nr': mr['cov_nr'],
                'arm': mr['arm'], 'run_a_na': is_na_a, 'run_b_na': is_na_b,
            })

    if paired:
        result = compute_agreement(field_name, bucket, paired)
        results.append(result)

print(f"Fields analyzed: {len(results)}")


Fields analyzed: 59


## Section 3: Build and save per-field table

Build a results table with primary and secondary agreement metrics
per field. Fields below threshold are flagged for review. The table
is saved to `table_S1_consistency.csv`. Flagged fields are printed
sorted by primary metric (worst-performing first).


In [4]:
# Build per-field results table with primary and secondary agreement metrics.
# Fields below threshold are flagged for review.
rows = []
for r in results:
    n_total = r.n_compared + r.n_both_na + r.n_one_na
    rows.append({
        'field_name': r.field_name, 'bucket': r.bucket,
        'n_total': n_total, 'n_compared': r.n_compared,
        'n_both_na': r.n_both_na, 'n_one_na': r.n_one_na,
        'primary_metric': r.primary_metric_name,
        'primary_value': round(r.primary_metric_value, 4) if r.primary_metric_value is not None else None,
        'secondary_metric': r.secondary_metric_name,
        'secondary_value': round(r.secondary_metric_value, 4) if r.secondary_metric_value is not None else None,
        'tertiary_metric': r.tertiary_metric_name,
        'tertiary_value': round(r.tertiary_metric_value, 4) if r.tertiary_metric_value is not None else None,
        'flagged': r.flagged,
        'flag_reason': r.flag_reason if r.flagged else '',
    })

per_field = pd.DataFrame(rows)
print(f"Fields: {len(per_field)}, Flagged: {per_field['flagged'].sum()}")

flagged = per_field[per_field['flagged']].sort_values('primary_value')
print("\nFlagged fields:")
# Shorten secondary_metric names for compact display.
flagged['secondary_metric_abbr'] = flagged['secondary_metric'].map({
    'percent_agreement': 'pct_agree',
    'na_concordance_rate': 'na_concord',
    'value_agreement_on_matched_keys': 'val_agree',
})
print(flagged[['field_name', 'bucket', 'primary_metric', 'primary_value', 'secondary_metric_abbr', 'secondary_value']].to_string())

per_field.to_csv(OUTPUT_DIR / 'pipeline' / 'table_S1_consistency.csv',
        index=False)
print("Saved: table_S1_consistency.csv")


Fields: 59, Flagged: 26

Flagged fields:
                                   field_name                   bucket primary_metric  primary_value secondary_metric_abbr  secondary_value
50                    digital_literacy_skills  structured_string_array    key_jaccard         0.2000             val_agree           0.0000
36            health_literacy_instrument_name              categorical       gwet_ac1         0.3956             pct_agree           0.4444
52                                 ses_income  structured_string_array    key_jaccard         0.4952             val_agree           0.0000
48                digital_literacy_possession  structured_string_array    key_jaccard         0.5000             val_agree           1.0000
53                       ses_living_situation  structured_string_array    key_jaccard         0.6036             val_agree           0.6923
33              healthcare_setting_confidence              categorical       gwet_ac1         0.6154             pct_ag

## Section 4: Per-trial one-NA flags

Collect cases where exactly one LLM run returned NA for a field-trial-arm
combination (the other run extracted a value). These are saved to
`inter_run_ambiguous_flags.csv` and fed into N03's `classify_equity_reporting()`
as `ambiguous_cov_nrs` — trials flagged here get labeled "ambiguous" rather
than "not_reported" in PROGRESS-Plus reporting, distinguishing "field
definitely absent" from "LLM might have missed it."

Only PROGRESS-Plus equity fields are reported (non-equity one-NA cases
are collected but not broken down by field).


In [5]:
# Identify one-NA cases: exactly one run extracted data, the other returned NA.
# Feeds N03 as 'ambiguous' flags — unclear if field absent or LLM missed it.
one_na_cases_df = pd.DataFrame(per_trial_na)
print(f"Total one-NA cases: {len(one_na_cases_df)}")

progress_plus_fields = [
    'ses',
    'ses_income', 'ses_living_situation', 'ses_relationship_status',
    'ses_job_status', 'ses_living_location', 'educational_level', 'ethnicity',
    'health_literacy', 'digital_literacy',
]
progress_plus_one_na = one_na_cases_df[one_na_cases_df['field_name'].isin(progress_plus_fields)]
print(f"PROGRESS-Plus one-NA cases: {len(progress_plus_one_na)}")

for field in progress_plus_fields:
    subset = progress_plus_one_na[progress_plus_one_na['field_name'] == field]
    if len(subset) > 0:
        cov_list = sorted(subset['cov_nr'].unique())
        print(f"  {field}: {len(subset)} cases, trials={cov_list}")

one_na_cases_df.to_csv(OUTPUT_DIR / 'pipeline' / 'inter_run_ambiguous_flags.csv', index=False)
print("Saved: inter_run_ambiguous_flags.csv")


Total one-NA cases: 186
PROGRESS-Plus one-NA cases: 8
  ses_income: 4 cases, trials=['4475', '4724']
  ses_living_situation: 2 cases, trials=['4986']
  ses_relationship_status: 2 cases, trials=['4986']
Saved: inter_run_ambiguous_flags.csv


In [6]:
print("=== Notebook 01 complete ===")
print(f"Completed at: {pd.Timestamp.now().isoformat()}")
print(
    f"Analyzed: {len(per_field)} fields",
    f"{per_field['flagged'].sum()} flagged"
)

print(f"Schema hash: 2b702021 (this revision's reference)")
# Schema staleness check: compare computed hash against stored version.
# If column schemas changed since last run, downstream notebooks may
# produce stale results.
from src.analysis.data_loading import get_schema_hash
current_hash = get_schema_hash()
version_file = ROOT / "data" / "processed" / "schema_version.txt"
if version_file.exists():
    stored_hash = version_file.read_text().strip()
    if current_hash != stored_hash:
        print(f"WARNING: Schema hash changed from {stored_hash} to {current_hash} — downstream data may be stale")
    else:
        print(f"Schema hash verified: {current_hash}")
else:
    print(f"Schema hash: {current_hash} (first run — no prior hash stored)")
version_file.write_text(current_hash)


=== Notebook 01 complete ===
Completed at: 2026-06-14T17:07:21.741960
Analyzed: 59 fields 26 flagged
Schema hash: 2b702021 (this revision's reference)
Schema hash verified: 2b702021


8